In [ ]:
import copy
import json
import os
import tempfile
import time
from collections import defaultdict
from typing import Dict, List, Set

import numpy as np
import zstandard as zstd

from pale.hashing import hash_chunk as _hash
from pale.chunking import chunk_bytes as _chunk
from pale.serialization import tensor_to_bytes as _tensor_to_bytes


def _to_bytes(arr: np.ndarray) -> bytes:
    raw, _, _ = _tensor_to_bytes(arr)
    return raw


_cctx = zstd.ZstdCompressor(level=3)
CHUNK_SIZE = 256 * 1024  # 256 KB


def _extract_sklearn(model) -> Dict[str, np.ndarray]:
    tensors = {}
    for i, col in enumerate(model.estimators_):
        for j, tree in enumerate(col):
            idx = i * len(col) + j
            t = tree.tree_
            tensors[f"tree_{idx:06d}_features"] = t.feature.astype(np.int32)
            tensors[f"tree_{idx:06d}_thresholds"] = t.threshold.astype(np.float64)
            tensors[f"tree_{idx:06d}_values"] = t.value.squeeze().astype(np.float64)
    return dict(sorted(tensors.items()))


def _extract_xgboost(booster) -> Dict[str, np.ndarray]:
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
        tmp = f.name
    try:
        booster.save_model(tmp)
        with open(tmp) as f:
            model_json = json.load(f)
    finally:
        os.unlink(tmp)
    trees = model_json["learner"]["gradient_booster"]["model"].pop("trees")
    skeleton_bytes = json.dumps(model_json).encode()
    tensors = {"__skeleton__": np.frombuffer(skeleton_bytes, dtype=np.uint8).copy()}
    for i, tree in enumerate(trees):
        tb = json.dumps(tree).encode()
        tensors[f"tree_{i:06d}"] = np.frombuffer(tb, dtype=np.uint8).copy()
    return dict(sorted(tensors.items()))


def _extract_pytorch(state_dict) -> Dict[str, np.ndarray]:
    import torch

    tensors = {}
    for k, v in state_dict.items():
        arr = v.detach().cpu().numpy()
        tensors[k] = np.ascontiguousarray(arr)
    return dict(sorted(tensors.items()))


# ── training helpers ──────────────────────────────────────────────────────────


def _train_sklearn_sequence(n_steps: int = 10, trees_per_step: int = 10):
    from sklearn.ensemble import GradientBoostingClassifier

    rng = np.random.default_rng(0)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(int)
    model = GradientBoostingClassifier(
        n_estimators=trees_per_step, warm_start=True, random_state=0
    )
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models


def _train_sklearn_run2(n_steps: int = 10, trees_per_step: int = 10):
    from sklearn.ensemble import GradientBoostingClassifier

    rng = np.random.default_rng(99)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(int)
    model = GradientBoostingClassifier(
        n_estimators=trees_per_step, warm_start=True, random_state=99
    )
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models


def _train_xgboost_sequence(n_steps: int = 10, rounds_per_step: int = 10):
    import xgboost as xgb

    rng = np.random.default_rng(0)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(np.float32)
    dtrain = xgb.DMatrix(X, label=y)
    params = {"max_depth": 3, "objective": "binary:logistic", "seed": 0, "verbosity": 0}
    boosters, booster = [], None
    for _ in range(n_steps):
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=rounds_per_step,
            xgb_model=booster,
            verbose_eval=False,
        )
        boosters.append(booster)
    return boosters


def _train_xgboost_run2(n_steps: int = 10, rounds_per_step: int = 10):
    import xgboost as xgb

    rng = np.random.default_rng(99)
    X = rng.standard_normal((500, 10)).astype(np.float32)
    y = (X[:, 0] > 0).astype(np.float32)
    dtrain = xgb.DMatrix(X, label=y)
    params = {
        "max_depth": 3,
        "objective": "binary:logistic",
        "seed": 99,
        "verbosity": 0,
    }
    boosters, booster = [], None
    for _ in range(n_steps):
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=rounds_per_step,
            xgb_model=booster,
            verbose_eval=False,
        )
        boosters.append(booster)
    return boosters


def _make_pt_model(device):
    import torch.nn as nn

    return nn.Sequential(
        nn.Linear(64, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    ).to(device)


def _train_pytorch_base(device, n_epochs: int = 5):
    import torch
    import torch.nn as nn

    rng = np.random.default_rng(0)
    X = torch.from_numpy(rng.standard_normal((1000, 64)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, 10, 1000).astype(np.int64)).to(device)
    model = _make_pt_model(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    ds = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(0),
        num_workers=0,
    )
    model.train()
    for _ in range(n_epochs):
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return {k: v.clone() for k, v in model.state_dict().items()}


def _train_pytorch_finetune(base_state, seed: int, device, n_epochs: int = 5):
    """Fine-tune from base_state. Returns only the fine-tuned checkpoints (not base).

    The base is excluded so cross-run measurement reflects divergence of fine-tuned
    weights only. The caller passes base_state into measure_crossrun separately if
    shared-base chunk overlap is what's being measured.
    """
    import torch
    import torch.nn as nn

    rng = np.random.default_rng(seed)
    X = torch.from_numpy(rng.standard_normal((1000, 64)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, 10, 1000).astype(np.int64)).to(device)
    model = _make_pt_model(device)
    model.load_state_dict(base_state)
    opt = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    ds = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
    )
    state_dicts = []
    model.train()
    for _ in range(n_epochs):
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})
    return state_dicts


def _train_pytorch_sequence(device, n_epochs: int = 20, freeze_epoch: int = 10):
    import torch
    import torch.nn as nn

    rng = np.random.default_rng(42)
    X = torch.from_numpy(rng.standard_normal((1000, 64)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, 10, 1000).astype(np.int64)).to(device)
    model = _make_pt_model(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    ds = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(0),
        num_workers=0,
    )
    frozen = False
    state_dicts = []
    model.train()
    for epoch in range(1, n_epochs + 1):
        if epoch == freeze_epoch + 1 and not frozen:
            # Freeze all layers except the last Linear (model[4])
            for layer in [model[0], model[2]]:
                for p in layer.parameters():
                    p.requires_grad_(False)
            opt = torch.optim.SGD(
                [p for p in model.parameters() if p.requires_grad],
                lr=0.001,
                momentum=0.9,
            )
            frozen = True
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})
    return state_dicts


# ── measurements ─────────────────────────────────────────────────────────────


def measure_noop(sequences: List[Dict[str, np.ndarray]]) -> Dict:
    tensor_names = sorted(sequences[0].keys())
    stats = {
        n: {"identical": 0, "changed": 0, "compressed_bytes": []} for n in tensor_names
    }
    for prev, curr in zip(sequences, sequences[1:]):
        for name in tensor_names:
            if name not in curr:
                continue
            raw_p = _to_bytes(prev[name])
            raw_c = _to_bytes(curr[name])
            if _hash(raw_p) == _hash(raw_c):
                stats[name]["identical"] += 1
            else:
                stats[name]["changed"] += 1
                stats[name]["compressed_bytes"].append(len(_cctx.compress(raw_c)))
    # For tensors that never changed, record compressed size of first version as baseline
    for name in tensor_names:
        if not stats[name]["compressed_bytes"]:
            raw = _to_bytes(sequences[0][name])
            stats[name]["compressed_bytes"].append(len(_cctx.compress(raw)))
    n_pairs = len(sequences) - 1
    result = {}
    for name, s in stats.items():
        avg_c = sum(s["compressed_bytes"]) / len(s["compressed_bytes"])
        result[name] = {
            "identical": s["identical"],
            "changed": s["changed"],
            "identical_pct": s["identical"] / n_pairs * 100,
            # For frozen tensors this is the base size; for changed tensors it's avg of changed versions
            "avg_compressed_bytes": avg_c,
        }
    all_id = sum(s["identical"] for s in stats.values())
    all_tot = n_pairs * len(tensor_names)
    result["__summary__"] = {
        "identical_pct": all_id / all_tot * 100 if all_tot else 0.0,
    }
    return result


def measure_chunk_dedup(
    sequences: List[Dict[str, np.ndarray]], chunk_size: int
) -> Dict:
    """Measure chunk reuse against the cumulative store, not just the previous step.

    For each changed tensor at step N, checks whether any of its chunks already
    exist in the CAS — meaning any chunk seen at steps 1..N-1. This correctly
    models what a real CAS would find: a chunk present at step 1 and step 5
    counts as reused at step 5 even if steps 2-4 didn't contain it.
    """
    tensor_names = sorted(sequences[0].keys())
    changed: Dict[str, Dict] = defaultdict(lambda: {"reused": 0, "total": 0})
    cumulative_hashes: Dict[str, Set[str]] = defaultdict(set)

    for prev, curr in zip(sequences, sequences[1:]):
        for name in tensor_names:
            if name not in curr:
                continue
            raw_p = _to_bytes(prev[name])
            raw_c = _to_bytes(curr[name])
            # Add prev chunks to cumulative store before checking curr
            for c in _chunk(raw_p, chunk_size):
                cumulative_hashes[name].add(_hash(c))
            if _hash(raw_p) == _hash(raw_c):
                continue
            for c in _chunk(raw_c, chunk_size):
                changed[name]["total"] += 1
                if _hash(c) in cumulative_hashes[name]:
                    changed[name]["reused"] += 1

    result = {}
    tot_r, tot_t = 0, 0
    for name, s in changed.items():
        result[name] = {
            "total": s["total"],
            "reused": s["reused"],
            "reuse_pct": s["reused"] / s["total"] * 100 if s["total"] else 0.0,
        }
        tot_r += s["reused"]
        tot_t += s["total"]
    result["__summary__"] = {
        "reuse_pct": tot_r / tot_t * 100 if tot_t else 0.0,
        "no_changed_tensors": len(changed) == 0,
    }
    return result


def measure_crossrun(
    seqs_r1: List[Dict[str, np.ndarray]],
    seqs_r2: List[Dict[str, np.ndarray]],
    chunk_size: int,
) -> Dict:
    def _unique_hashes(seqs) -> Set[str]:
        s: Set[str] = set()
        for tensors in seqs:
            for arr in tensors.values():
                for c in _chunk(_to_bytes(arr), chunk_size):
                    s.add(_hash(c))
        return s

    h1 = _unique_hashes(seqs_r1)
    h2 = _unique_hashes(seqs_r2)
    shared = len(h1 & h2)
    return {
        "run1_unique": len(h1),
        "run2_unique": len(h2),
        "shared": shared,
        "new_in_run2": len(h2) - shared,
        "reuse_pct": shared / len(h2) * 100 if h2 else 0.0,
    }


# ── printing ──────────────────────────────────────────────────────────────────


def _fmt_bytes(n: float) -> str:
    size = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024:
            return f"{size:.1f}{unit}"
        size /= 1024
    return f"{size:.1f}TB"


def print_noop(label: str, stats: Dict) -> None:
    summary = stats.get("__summary__", {})
    print(f"\n[{label}] No-op fast path — tensor identity across steps")
    print(f"  Overall: {summary['identical_pct']:.1f}% of tensor-steps byte-identical")
    print(
        f"  {'Tensor':<36} {'Identical':>10} {'Changed':>8} {'Identical%':>11} {'AvgCompressed':>14}"
    )
    print(f"  {'-' * 36} {'-' * 10} {'-' * 8} {'-' * 11} {'-' * 14}")
    for name, s in sorted((k, v) for k, v in stats.items() if k != "__summary__"):
        print(
            f"  {name:<36} {s['identical']:>10} {s['changed']:>8} "
            f"{s['identical_pct']:>10.1f}% {_fmt_bytes(s['avg_compressed_bytes']):>14}"
        )


def print_chunk(label: str, stats: Dict, chunk_size: int) -> None:
    cs = (
        f"{chunk_size // 1024}KB"
        if chunk_size < 1024**2
        else f"{chunk_size // 1024**2}MB"
    )
    summary = stats.get("__summary__", {})
    print(f"\n[{label}] Chunk-level reuse in changed tensors (chunk={cs})")
    if summary.get("no_changed_tensors"):
        print("  Overall: N/A — no changed tensors")
        return
    print(
        f"  Overall: {summary['reuse_pct']:.1f}% of chunks in changed tensors already in store"
    )
    print(f"  {'Tensor':<36} {'Total':>8} {'Reused':>8} {'Reuse%':>8}")
    print(f"  {'-' * 36} {'-' * 8} {'-' * 8} {'-' * 8}")
    for name, s in sorted((k, v) for k, v in stats.items() if k != "__summary__"):
        print(f"  {name:<36} {s['total']:>8} {s['reused']:>8} {s['reuse_pct']:>7.1f}%")


def print_crossrun(label: str, stats: Dict) -> None:
    print(f"\n[{label}] Cross-run chunk sharing")
    print(f"  Run 1 unique chunks : {stats['run1_unique']:,}")
    print(f"  Run 2 unique chunks : {stats['run2_unique']:,}")
    print(f"  Shared (R1 ∩ R2)    : {stats['shared']:,}  ({stats['reuse_pct']:.1f}%)")
    print(f"  New in run 2        : {stats['new_in_run2']:,}")


# ── main ──────────────────────────────────────────────────────────────────────


def run_sklearn(n_steps: int = 10, trees_per_step: int = 10) -> None:
    print("\n" + "=" * 60)
    print("SKLEARN — GradientBoostingClassifier warm-start")
    print("=" * 60)

    t0 = time.time()
    print("  Training run 1...", end=" ", flush=True)
    models_r1 = _train_sklearn_sequence(n_steps, trees_per_step)
    print(f"{time.time() - t0:.1f}s")

    seqs_r1 = [_extract_sklearn(m) for m in models_r1]
    print_noop("sklearn", measure_noop(seqs_r1))
    print_chunk("sklearn", measure_chunk_dedup(seqs_r1, CHUNK_SIZE), CHUNK_SIZE)

    t0 = time.time()
    print("\n  Training run 2 (seed=99)...", end=" ", flush=True)
    models_r2 = _train_sklearn_run2(n_steps, trees_per_step)
    print(f"{time.time() - t0:.1f}s")

    seqs_r2 = [_extract_sklearn(m) for m in models_r2]
    print_crossrun(
        "sklearn (seed=0 vs seed=99)", measure_crossrun(seqs_r1, seqs_r2, CHUNK_SIZE)
    )


def run_xgboost(n_steps: int = 10, rounds_per_step: int = 10) -> None:
    print("\n" + "=" * 60)
    print("XGBOOST — warm-start boosting")
    print("=" * 60)

    t0 = time.time()
    print("  Training run 1...", end=" ", flush=True)
    boosters_r1 = _train_xgboost_sequence(n_steps, rounds_per_step)
    print(f"{time.time() - t0:.1f}s")

    seqs_r1 = [_extract_xgboost(b) for b in boosters_r1]
    print_noop("xgboost", measure_noop(seqs_r1))
    print_chunk("xgboost", measure_chunk_dedup(seqs_r1, CHUNK_SIZE), CHUNK_SIZE)

    t0 = time.time()
    print("\n  Training run 2 (seed=99)...", end=" ", flush=True)
    boosters_r2 = _train_xgboost_run2(n_steps, rounds_per_step)
    print(f"{time.time() - t0:.1f}s")

    seqs_r2 = [_extract_xgboost(b) for b in boosters_r2]
    print_crossrun(
        "xgboost (seed=0 vs seed=99)", measure_crossrun(seqs_r1, seqs_r2, CHUNK_SIZE)
    )


def run_pytorch(
    device: str = "cpu", n_epochs: int = 20, freeze_epoch: int = 10, n_finetune: int = 5
) -> None:
    print("\n" + "=" * 60)
    print(f"PYTORCH — MLP fine-tuning (device={device})")
    print("=" * 60)

    t0 = time.time()
    print(
        f"  Training {n_epochs}-epoch sequence (freeze after epoch {freeze_epoch})...",
        end=" ",
        flush=True,
    )
    state_dicts = _train_pytorch_sequence(device, n_epochs, freeze_epoch)
    print(f"{time.time() - t0:.1f}s")

    seqs = [_extract_pytorch(sd) for sd in state_dicts]
    print_noop("pytorch", measure_noop(seqs))
    print_chunk("pytorch", measure_chunk_dedup(seqs, CHUNK_SIZE), CHUNK_SIZE)

    # Cross-run: measure fine-tuned weights only (base excluded).
    # Both runs start from the same base_state so base chunks would inflate shared
    # count trivially — we measure divergence of the fine-tuned weights here.
    t0 = time.time()
    print("\n  Training shared base + 2 fine-tune runs...", end=" ", flush=True)
    base_state = _train_pytorch_base(device)
    sds_r1 = _train_pytorch_finetune(
        base_state, seed=42, device=device, n_epochs=n_finetune
    )
    sds_r2 = _train_pytorch_finetune(
        base_state, seed=99, device=device, n_epochs=n_finetune
    )
    print(f"{time.time() - t0:.1f}s")

    seqs_r1 = [_extract_pytorch(sd) for sd in sds_r1]
    seqs_r2 = [_extract_pytorch(sd) for sd in sds_r2]
    print_crossrun(
        "pytorch (fine-tuned only, seed=42 vs seed=99)",
        measure_crossrun(seqs_r1, seqs_r2, CHUNK_SIZE),
    )


if __name__ == "__main__":
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    run_sklearn()
    run_xgboost()
    run_pytorch(device=device)
    print("\n=== done ===")
